In [ ]:
import pandas as pd

RUTA = "../fuentes"

In [2]:
prod = pd.read_excel(f"{RUTA}/fuente_produccion_campo.xlsx", dtype = str)
ven = pd.read_excel(f"{RUTA}/fuente_ventas_pedidos.xlsx", dtype = str)
log = pd.read_excel(f"{RUTA}/fuente_logistica_despachos.xlsx", dtype = str)

In [3]:
print("Producción:", len(prod), "filas")
print("Ventas:    ", len(ven), "filas")
print("Logística: ", len(log), "filas")
prod.head()

Producción: 462 filas
Ventas:     515 filas
Logística:  493 filas


,Lote,Fecha Cosecha,Cultivo,Variedad,Mercado Destino,Hectáreas,Kg Ingresados,Kg Descartados,Ton Conformes
0,LT-00001,01-07-25,Arándano,Ventura,ALEMANIA,10.5,103050,8520,94.53
1,LT-00002,2025-07-01 00:00:00,PALTA,Hass,Alemania,8.9,87430,4650,82.78
2,LT-00003,2025-07-01 00:00:00,Palta,Hass,china,3.8,35840,NaN,33.06
3,LT-00004,2025-07-02 00:00:00,Arándano,Ventura,Holanda,6.4,65290,4180,61.11
4,LT-00005,2025-07-03 00:00:00,Arándano,Biloxi,Reino Unido,9.7,104880,9440,95.44


In [4]:
prod.columns = ["lote", "fecha_cosecha", "cultivo", "variedad", "mercado_destino", "hectareas", "kg_ingresados", "kg_descartados", "ton_conformes"]
ven.columns  = ["nro_pedido", "fecha_pedido", "cliente", "pais_destino", "producto", "cantidad", "unidad", "precio_usd_kg"]
log.columns  = ["nro_pedido", "fecha_salida", "llegada_programada", "llegada_real", "puerto_destino", "kg_entregados", "reclamo"]

prod.head()

,lote,fecha_cosecha,cultivo,variedad,mercado_destino,hectareas,kg_ingresados,kg_descartados,ton_conformes
0,LT-00001,01-07-25,Arándano,Ventura,ALEMANIA,10.5,103050,8520,94.53
1,LT-00002,2025-07-01 00:00:00,PALTA,Hass,Alemania,8.9,87430,4650,82.78
2,LT-00003,2025-07-01 00:00:00,Palta,Hass,china,3.8,35840,NaN,33.06
3,LT-00004,2025-07-02 00:00:00,Arándano,Ventura,Holanda,6.4,65290,4180,61.11
4,LT-00005,2025-07-03 00:00:00,Arándano,Biloxi,Reino Unido,9.7,104880,9440,95.44


In [5]:
from sqlalchemy import create_engine

SERVIDOR = r"localhost\SQLEXPRESS"      
engine = create_engine(
    f"mssql+pyodbc://@{SERVIDOR}/DW_AgroVerde"
    "?driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes&TrustServerCertificate=yes",
    fast_executemany=True)

pd.read_sql("SELECT DB_NAME() AS base_de_datos", engine)

,base_de_datos
0,Dw_AgroVerde


In [6]:
for nombre, tabla in [("stg_produccion", prod), ("stg_ventas", ven), ("stg_despachos", log)]:
    tabla.astype("object").where(tabla.notna(), None).to_sql(nombre, engine, if_exists="replace", index=False)
    print("guardada", nombre, len(tabla))

guardada stg_produccion 462
guardada stg_ventas 515
guardada stg_despachos 493


### TRANSFORMACIÓN DE LA FUENTE DE DATOS DE PRODUCCIÓN

In [7]:
import re, unicodedata

def clave(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())

PAISES = {"paisesbajos":"Países Bajos","holanda":"Países Bajos","espana":"España",
          "reinounido":"Reino Unido","uk":"Reino Unido","alemania":"Alemania",
          "germany":"Alemania","china":"China","japon":"Japón"}

prod["pais"] = prod["mercado_destino"].map(lambda s: PAISES.get(clave(s)))
print(prod["pais"].value_counts(dropna=False))

pais
Países Bajos    88
China           83
España          80
Reino Unido     78
Alemania        76
Japón           57
Name: count, dtype: int64


In [8]:
cambiados = prod["mercado_destino"] != prod["pais"]
print("Registros afectados:", cambiados.sum())
prod.loc[cambiados, ["mercado_destino", "pais"]].drop_duplicates().head(8)

Registros afectados: 71


,mercado_destino,pais
0,ALEMANIA,Alemania
2,china,China
3,Holanda,Países Bajos
8,Espana,España
18,PAISES BAJOS,Países Bajos
23,ESPAÑA,España
36,CHINA,China
58,JAPÓN,Japón


In [9]:
import datetime as dt

def parse_date(s):
    if pd.isna(s): return pd.NaT
    for f in ("%Y-%m-%d %H:%M:%S", "%d/%m/%Y", "%Y-%m-%d", "%d-%m-%y"):
        try: return dt.datetime.strptime(str(s).strip(), f).date()
        except ValueError: pass
    return pd.NaT

prod["fecha"] = prod["fecha_cosecha"].map(parse_date)

cambiados = prod["fecha_cosecha"].notna() & ~prod["fecha_cosecha"].fillna("").str.match(r"^\d{4}-\d{2}-\d{2} 00:00:00$")
print("Registros afectados:", cambiados.sum(), "| fechas no reconocidas:", prod["fecha"].isna().sum())
prod.loc[cambiados, ["fecha_cosecha", "fecha"]].head(8)

Registros afectados: 108 | fechas no reconocidas: 0


,fecha_cosecha,fecha
0,01-07-25,2025-07-01
5,2025-07-04,2025-07-04
6,04/07/2025,2025-07-04
8,05/07/2025,2025-07-05
13,2025-07-10,2025-07-10
16,11-07-25,2025-07-11
20,15/07/2025,2025-07-15
27,21-07-25,2025-07-21


In [10]:
print("Filas antes:", len(prod))
n = prod.duplicated(subset="lote").sum()
prod = prod.drop_duplicates(subset="lote", keep="first")
print("Duplicados eliminados:", n, "| Filas después:", len(prod))

Filas antes: 462
Duplicados eliminados: 12 | Filas después: 450


In [ ]:
import numpy as np

def to_num(x):
    if pd.isna(x): return np.nan
    t = re.sub(r"[^0-9.\-]", "", str(x).replace(",", ""))
    return float(t) if t not in ("", "-", ".") else np.nan

for c in ["hectareas", "kg_ingresados", "kg_descartados", "ton_conformes"]:
    prod[c] = prod[c].map(to_num)

neg = prod["kg_ingresados"] < 0
prod.loc[neg, "kg_ingresados"] = prod.loc[neg, "kg_ingresados"].abs()
print("Negativos corregidos:", neg.sum())

m1 = prod["kg_descartados"].isna() & prod["ton_conformes"].notna()
m2 = prod["ton_conformes"].isna() & prod["kg_descartados"].notna()
prod.loc[m1, "kg_descartados"] = prod.loc[m1, "kg_ingresados"] - prod.loc[m1, "ton_conformes"] * 1000
prod.loc[m2, "ton_conformes"] = ((prod.loc[m2, "kg_ingresados"] - prod.loc[m2, "kg_descartados"]) / 1000).round(2)
print("Nulos imputados:", (m1 | m2).sum())

Negativos corregidos: 5
Nulos imputados: 27


In [ ]:
obligatorios = ["cultivo", "pais", "fecha", "hectareas", "kg_ingresados", "kg_descartados", "ton_conformes"]
r1 = prod[obligatorios].isna().any(axis=1)              
r2 = prod["hectareas"] <= 0                             
r3 = prod["kg_descartados"] > prod["kg_ingresados"]  
rechazo = r1 | r2 | r3

print("Faltan datos:", r1.sum(), "| Hectáreas ≤ 0:", r2.sum(), "| Descartados > ingresados:", r3.sum())
rechazados = prod[rechazo]          
prod = prod[~rechazo].copy()
print("Filas finales de Producción:", len(prod))

Faltan datos: 0 | Hectáreas ≤ 0: 4 | Descartados > ingresados: 3
Filas finales de Producción: 443


### TRANSFORMACIÓN DE LA FUENTE DE DATOS DE VENTAS

In [13]:
def reporte(regla, antes, despues, mask=None):
    if mask is None:
        mask = antes.fillna("").astype(str) != despues.fillna("").astype(str)
    print(f"{regla}: {mask.sum()} afectados")
    display(pd.DataFrame({"antes": antes[mask], "despues": despues[mask]}).drop_duplicates().head(3))

print("Filas iniciales Ventas:", len(ven))

antes = ven["fecha_pedido"].copy()
ven["fecha"] = antes.map(parse_date)
reporte("Fechas", antes, ven["fecha"], antes.notna() & ~antes.fillna("").str.match(r"^\d{4}-\d{2}-\d{2} 00:00:00$"))

antes = ven["pais_destino"].copy()
ven["pais"] = antes.map(lambda s: PAISES.get(clave(s)))
reporte("País", antes, ven["pais"])

antes = ven["cliente"].copy()
ven["cli_key"] = antes.map(clave)
frecuente = ven.assign(c=antes.str.strip()).groupby("cli_key")["c"].agg(lambda s: s.value_counts().idxmax())
ven["cliente"] = ven["cli_key"].map(frecuente)
reporte("Cliente", antes, ven["cliente"])

prod["variedad"] = prod["variedad"].str.strip()
CATALOGO = {clave(f"{c}{v}"): f"{c} {v}" for c, v in prod[["cultivo", "variedad"]].drop_duplicates().itertuples(index=False)}
antes = ven["producto"].copy()
ven["producto"] = antes.map(lambda s: CATALOGO.get(clave(s)))
reporte("Producto", antes, ven["producto"])

antes = ven["cantidad"].copy()
ven["cantidad_kg"] = antes.map(to_num)
ton = ven["unidad"].fillna("").str.strip().str.lower() == "ton"
ven.loc[ton, "cantidad_kg"] = ven.loc[ton, "cantidad_kg"] * 1000
reporte("Toneladas a kg", antes, ven["cantidad_kg"], ton)
neg = ven["cantidad_kg"] < 0
ven.loc[neg, "cantidad_kg"] = ven.loc[neg, "cantidad_kg"].abs()
print("Cantidades negativas corregidas:", neg.sum())

antes = ven["precio_usd_kg"].copy()
ven["precio"] = antes.map(to_num)
reporte("Precio texto a número", antes, ven["precio"], antes.fillna("").str.contains(r"[A-Za-z$]"))

n = ven.duplicated(subset="nro_pedido").sum()
ven = ven.drop_duplicates(subset="nro_pedido", keep="first")
print("Duplicados eliminados:", n)

ven["ym"] = ven["fecha"].map(lambda d: (d.year, d.month) if pd.notna(d) else None)
prom_mes = ven.groupby(["producto", "ym"])["precio"].transform("mean")
prom_prod = ven.groupby("producto")["precio"].transform("mean")
vacio = ven["precio"].isna()
ven.loc[vacio, "precio"] = prom_mes[vacio].fillna(prom_prod[vacio]).round(2)
print("Precios vacíos imputados:", vacio.sum())

rechazo = ven[["producto", "pais", "fecha", "precio", "cantidad_kg"]].isna().any(axis=1) | (ven["cantidad_kg"] <= 0)
rech_ven = ven[rechazo]
ven = ven[~rechazo].copy()
ven["pedido_key"] = ven["nro_pedido"].str.strip()
print("Rechazados:", len(rech_ven), "| Filas finales de Ventas:", len(ven))

Filas iniciales Ventas: 515
Fechas: 117 afectados


,antes,despues
8,2025-07-13,2025-07-13
10,16/07/2025,2025-07-16
17,23/07/2025,2025-07-23


País: 140 afectados


,antes,despues
0,PAISES BAJOS,Países Bajos
1,Paises Bajos,Países Bajos
3,Holanda,Países Bajos


Cliente: 38 afectados


...

Toneladas a kg: 29 afectados


,antes,despues
0,18,18000.0
39,10.5,10500.0
45,19,19000.0


Cantidades negativas corregidas: 2
Precio texto a número: 80 afectados


,antes,despues
4,$4.25,4.25
7,US$ 7.63,7.63
9,US$ 8.77,8.77


Duplicados eliminados: 12
Precios vacíos imputados: 39
Rechazados: 0 | Filas finales de Ventas: 503


### TRANSFORMACIÓN DE LA FUENTE DE DATOS DE DESPACHOS

In [ ]:
print("Filas iniciales Logística:", len(log))

antes = log["nro_pedido"].copy()
log["pedido_nro"] = antes.map(lambda s: f"PED-{int(re.sub(r'[^0-9]', '', str(s))):06d}")
reporte("N° de pedido", antes, log["pedido_nro"])

for c in ["fecha_salida", "llegada_programada", "llegada_real"]:
    log[c + "_d"] = log[c].map(parse_date)
print("Fechas no reconocidas:", log[["fecha_salida_d", "llegada_programada_d", "llegada_real_d"]].isna().sum().sum())

antes = log["kg_entregados"].copy()
log["kg_ent"] = antes.map(to_num)
reporte("kg con texto", antes, log["kg_ent"], antes.fillna("").str.contains(r"[A-Za-z]"))
print("kg entregados NULL (se mantienen, no se inventan):", log["kg_ent"].isna().sum())

antes = log["reclamo"].copy()
log["con_reclamo"] = antes.fillna("").map(lambda s: 1 if clave(s) in {"si", "s", "1"} else 0)
reporte("Reclamo a 1/0", antes, log["con_reclamo"], ~antes.fillna("").isin(["SI", "NO"]))

n = log.duplicated(subset="pedido_nro").sum()
log = log.drop_duplicates(subset="pedido_nro", keep="first")
print("Duplicados eliminados:", n)

r1 = log[["fecha_salida_d", "llegada_programada_d", "llegada_real_d"]].isna().any(axis=1)
r2 = pd.to_datetime(log["llegada_real_d"]) < pd.to_datetime(log["fecha_salida_d"])
print("Fecha ilegible:", r1.sum(), "| Llegada antes de salida:", r2.sum())
rech_log = log[r1 | r2]
log = log[~(r1 | r2)].copy()

cruce = ven[["pedido_key", "cliente", "producto", "pais", "cantidad_kg"]].rename(columns={"cantidad_kg": "cant_pedida"})
log = log.merge(cruce, left_on="pedido_nro", right_on="pedido_key", how="left")
huerfano = log["cliente"].isna()
print("Despachos sin pedido en Ventas:", huerfano.sum())
rech_log = pd.concat([rech_log, log[huerfano]])
log = log[~huerfano].copy()

log["dias_transito"] = (pd.to_datetime(log["llegada_real_d"]) - pd.to_datetime(log["fecha_salida_d"])).dt.days
log["entrega_a_tiempo"] = (pd.to_datetime(log["llegada_real_d"]) <= pd.to_datetime(log["llegada_programada_d"])).astype(int)
print("Rechazados:", len(rech_log), "| Filas finales de Logística:", len(log))

Filas iniciales Logística: 493
N° de pedido: 45 afectados


,antes,despues
1,ped000002,PED-000002
5,6,PED-000006
15,ped000016,PED-000016


Fechas no reconocidas: 0
kg con texto: 25 afectados


,antes,despues
2,18000 kg,18000.0
14,15000 kg,15000.0
32,13500 kg,13500.0


kg entregados NULL (se mantienen, no se inventan): 12
Reclamo a 1/0: 385 afectados


,antes,despues
0,No,0
4,Sí,1
6,no,0


Duplicados eliminados: 8
Fecha ilegible: 0 | Llegada antes de salida: 3
Despachos sin pedido en Ventas: 3
Rechazados: 6 | Filas finales de Logística: 479


In [15]:
CATEGORIA = {"Espárrago": "Hortaliza", "Arándano": "Fruta", "Palta": "Fruta"}
CONTINENTE = {"Países Bajos": "Europa", "España": "Europa", "Reino Unido": "Europa",
              "Alemania": "Europa", "China": "Asia", "Japón": "Asia"}

dim_producto = prod[["cultivo", "variedad"]].drop_duplicates().sort_values(["cultivo", "variedad"]).reset_index(drop=True)
dim_producto.insert(0, "id_producto", dim_producto.index + 1)
dim_producto["categoria"] = dim_producto["cultivo"].map(CATEGORIA)
dim_producto = dim_producto.rename(columns={"cultivo": "nombre"})[["id_producto", "nombre", "categoria", "variedad"]]
pid = {f"{r.nombre} {r.variedad}": r.id_producto for r in dim_producto.itertuples()}

clientes = sorted(ven["cliente"].unique())
dim_cliente = pd.DataFrame({"id_cliente": range(1, len(clientes) + 1), "razon_social": clientes})
cid = dict(zip(dim_cliente["razon_social"], dim_cliente["id_cliente"]))

puertos = log.groupby("pais")["puerto_destino"].agg(lambda s: s.value_counts().idxmax()).to_dict()
paises = sorted(set(ven["pais"]) | set(prod["pais"]))
dim_mercado = pd.DataFrame({"id_mercado": range(1, len(paises) + 1), "pais_destino": paises,
                            "continente": [CONTINENTE[p] for p in paises],
                            "puerto": [puertos.get(p, "N/D") for p in paises]})
mid = dict(zip(dim_mercado["pais_destino"], dim_mercado["id_mercado"]))

fechas = pd.concat([pd.Series(prod["fecha"]), pd.Series(ven["fecha"]), pd.Series(log["llegada_real_d"])]).dropna()
rng = pd.date_range(pd.to_datetime(fechas).min(), pd.to_datetime(fechas).max(), freq="D")
dim_tiempo = pd.DataFrame({"id_tiempo": rng.strftime("%Y%m%d").astype(int), "fecha": rng.date,
                           "dia": rng.day, "mes": rng.month, "trimestre": rng.quarter, "anio": rng.year})
tid = lambda s: pd.to_datetime(s).dt.strftime("%Y%m%d").astype(int)

fact_produccion = pd.DataFrame({
    "id_tiempo": tid(prod["fecha"]),
    "id_producto": (prod["cultivo"] + " " + prod["variedad"]).map(pid),
    "id_mercado": prod["pais"].map(mid),
    "lote_id": prod["lote"].str.strip(),
    "kg_ingresados": prod["kg_ingresados"].round(2),
    "kg_descartados": prod["kg_descartados"].round(2),
    "ton_conformes": prod["ton_conformes"].round(2),
    "hectareas_cosechadas": prod["hectareas"].round(2)})
fact_produccion.insert(0, "id_produccion", range(1, len(fact_produccion) + 1))

fact_pedido = pd.DataFrame({
    "id_tiempo": tid(ven["fecha"]), "id_producto": ven["producto"].map(pid),
    "id_cliente": ven["cliente"].map(cid), "id_mercado": ven["pais"].map(mid),
    "pedido_nro": ven["pedido_key"], "cantidad_pedida_kg": ven["cantidad_kg"].round(2),
    "precio_unitario_usd": ven["precio"].round(2)})
fact_pedido.insert(0, "id_pedido", range(1, len(fact_pedido) + 1))

fact_despacho = pd.DataFrame({
    "id_tiempo": tid(log["llegada_real_d"]), "id_producto": log["producto"].map(pid),
    "id_cliente": log["cliente"].map(cid), "id_mercado": log["pais"].map(mid),
    "pedido_nro": log["pedido_nro"], "cantidad_pedida_kg": log["cant_pedida"].round(2),
    "cantidad_entregada_kg": log["kg_ent"].round(2), "dias_transito": log["dias_transito"],
    "entrega_a_tiempo": log["entrega_a_tiempo"], "con_reclamo": log["con_reclamo"]})
fact_despacho.insert(0, "id_despacho", range(1, len(fact_despacho) + 1))

tablas = {"Dim_Tiempo": dim_tiempo, "Dim_Producto": dim_producto, "Dim_Cliente": dim_cliente,
          "Dim_Mercado": dim_mercado, "Fact_Produccion": fact_produccion,
          "Fact_Pedido": fact_pedido, "Fact_Despacho": fact_despacho}
for nombre, t in tablas.items():
    print(f"{nombre:<16} {len(t):>5} filas | ids vacíos: {t.filter(like='id_').isna().sum().sum()}")

Dim_Tiempo         394 filas | ids vacíos: 0
Dim_Producto        24 filas | ids vacíos: 0
Dim_Cliente         22 filas | ids vacíos: 0
Dim_Mercado          6 filas | ids vacíos: 0
Fact_Produccion    443 filas | ids vacíos: 0
Fact_Pedido        503 filas | ids vacíos: 0
Fact_Despacho      479 filas | ids vacíos: 0


In [16]:
CULTIVOS = {"esparrago": "Espárrago", "arandano": "Arándano", "palta": "Palta"}

antes = prod["cultivo"].copy()
prod["cultivo"] = antes.map(lambda s: CULTIVOS.get(clave(s)))
cambiados = antes != prod["cultivo"]
print("Cultivo - registros afectados:", cambiados.sum())
display(pd.DataFrame({"antes": antes[cambiados], "despues": prod.loc[cambiados, "cultivo"]}).drop_duplicates().head(5))

CATALOGO = {clave(f"{c}{v}"): f"{c} {v}" for c, v in prod[["cultivo", "variedad"]].drop_duplicates().itertuples(index=False)}
ven["producto"] = ven["producto"].map(lambda s: CATALOGO.get(clave(s)))
log["producto"] = log["producto"].map(lambda s: CATALOGO.get(clave(s)))
print("Combinaciones cultivo + variedad:", len(CATALOGO), "| productos sin reconocer:", ven["producto"].isna().sum() + log["producto"].isna().sum())

Cultivo - registros afectados: 66


,antes,despues
1,PALTA,Palta
10,espárrago,Espárrago
12,Arandano,Arándano
37,Esparrago,Espárrago
39,ARÁNDANO,Arándano


Combinaciones cultivo + variedad: 5 | productos sin reconocer: 0


In [ ]:
from sqlalchemy import text

ORDEN_BORRADO = ["Fact_Produccion", "Fact_Pedido", "Fact_Despacho", "Dim_Tiempo", "Dim_Producto", "Dim_Cliente", "Dim_Mercado"]
ORDEN_CARGA   = ["Dim_Tiempo", "Dim_Producto", "Dim_Cliente", "Dim_Mercado", "Fact_Produccion", "Fact_Pedido", "Fact_Despacho"]

with engine.begin() as con:                 
    for t in ORDEN_BORRADO:                
        con.execute(text(f"DELETE FROM {t}"))
    for t in ORDEN_CARGA:                   
        tablas[t].to_sql(t, con, if_exists="append", index=False)
        print(f"cargada {t:<16} {len(tablas[t]):>5} filas")

cargada Dim_Tiempo         394 filas
cargada Dim_Producto         5 filas
cargada Dim_Cliente         22 filas
cargada Dim_Mercado          6 filas
cargada Fact_Produccion    443 filas
cargada Fact_Pedido        503 filas
cargada Fact_Despacho      479 filas
